# Classical Baseline — TF-IDF + Linear Classifiers
### SEA 820 · GenDetect · Part 2 (Member A)

**Goal:** establish the honest **baseline score** the DistilBERT model must beat, on the
*exact same* test set produced by `data_processing/data_split.py`.

**Design (from EDA + TODO):**
- Input = the pre-computed **`text_classical`** column (heavy-cleaned, lemmatized) — no re-running the pipeline.
- TF-IDF is **fit on train only**, then transforms val/test (no leakage).
- Model selection happens on the **validation** set; the **test** set is touched **once**, at the end.
- Headline metric = **macro-F1** + per-class recall, because the data is imbalanced (63% human / 37% AI).
- We compare **default vs `class_weight='balanced'`**, and add **length features** as an engineering bonus.

> **Runtime:** Google Colab. Upload `metrics.py` and `tfidf_features.py` into your Drive
> `NLP_project/` folder (next to `preprocessing.py`) so the import below works.

## 0. Setup — mount Drive & import shared modules

In [ ]:
# Mount Google Drive (holds the split parquet files + the shared .py modules)
from google.colab import drive
drive.mount('/content/drive')

import os, sys, time
import numpy as np
import pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/NLP_project'
PROC_DIR  = os.path.join(DRIVE_DIR, 'processed')     # train/val/test.parquet live here
RESULTS   = os.path.join(DRIVE_DIR, 'classical_results')
os.makedirs(RESULTS, exist_ok=True)

# Import the shared project modules (uploaded next to preprocessing.py on Drive).
sys.path.append(DRIVE_DIR)
import tfidf_features as tf
import metrics as M

print('Drive dir :', DRIVE_DIR)
print('Processed :', os.listdir(PROC_DIR) if os.path.isdir(PROC_DIR) else 'NOT FOUND')
print('Results -> ', RESULTS)

## 1. Load the canonical splits
We load the same train/val/test the Transformer will use. We model on **`text_classical`**;
we keep the raw **`text`** around only for length features and error analysis.

In [ ]:
def load_split(name):
    path = os.path.join(PROC_DIR, f'{name}.parquet')
    df = pd.read_parquet(path)
    return df

train = load_split('train')
val   = load_split('val')
test  = load_split('test')

for name, d in [('train', train), ('val', val), ('test', test)]:
    bal = d['label'].value_counts(normalize=True).sort_index()
    print(f'{name:5s}: {len(d):>7,} rows | human {bal.get(0,0):.1%} / AI {bal.get(1,0):.1%}')

print('\ncolumns:', train.columns.tolist())
# Guard: text_classical must exist (built in Part 1). If missing, re-run data_split.py.
assert 'text_classical' in train.columns, 'text_classical column missing — re-run data_split.py'

## 2. Build TF-IDF features (fit on TRAIN only)
`tfidf_features.fit_transform` fits the vectorizer on train and only *transforms* val/test,
so no test vocabulary/IDF leaks into training.

In [ ]:
t0 = time.time()

vec, X_train, X_val, X_test = tf.fit_transform(
    train['text_classical'], val['text_classical'], test['text_classical'],
    max_features=50000, ngram_range=(1, 2), min_df=5, max_df=0.9,
)

y_train, y_val, y_test = train['label'].values, val['label'].values, test['label'].values

print(f'fit+transform in {time.time()-t0:.0f}s')
print(f'X_train: {X_train.shape}  (docs x features)')
print(f'X_val  : {X_val.shape}')
print(f'X_test : {X_test.shape}')
density = X_train.nnz / (X_train.shape[0] * X_train.shape[1])
print(f'vocabulary size : {len(vec.get_feature_names_out()):,}')
print(f'matrix density  : {density*100:.3f}%  (sparse -> memory-cheap)')

## 3. Baseline — Logistic Regression
Start simple (workshop week 3 pattern). We evaluate on **validation** — the test set stays sealed.
`liblinear` is a good fit for high-dimensional sparse text.

In [ ]:
from sklearn.linear_model import LogisticRegression

results_val = {}   # collect val metrics for every model we try

lr = LogisticRegression(max_iter=1000, C=1.0, solver='liblinear', random_state=42)
lr.fit(X_train, y_train)
results_val['LogReg (default)'] = M.print_metrics('LogReg (default) — VAL', y_val, lr.predict(X_val))

## 4. Class-imbalance experiment — default vs `class_weight='balanced'`
The TODO/EDA decision: only reweight the **training** set (val/test stay at the natural 63/37).
`balanced` makes the minority **AI** class count more, which usually **raises AI recall** — we
check whether that helps or hurts macro-F1.

In [ ]:
lr_bal = LogisticRegression(max_iter=1000, C=1.0, solver='liblinear',
                            class_weight='balanced', random_state=42)
lr_bal.fit(X_train, y_train)
results_val['LogReg (balanced)'] = M.print_metrics(
    'LogReg (class_weight=balanced) — VAL', y_val, lr_bal.predict(X_val))

print('\nTakeaway: compare AI recall and macro-F1 above. `balanced` trades a little')
print('human precision for higher AI recall — keep it only if macro-F1 improves.')

## 5. Stretch — Naive Bayes & Linear SVM
For top marks the rubric wants **2+ classic models compared**. Both are fast on sparse TF-IDF.
- **MultinomialNB** — classic text baseline, very fast.
- **LinearSVC** — often the strongest linear text classifier.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

nb = MultinomialNB()
nb.fit(X_train, y_train)
results_val['MultinomialNB'] = M.print_metrics('MultinomialNB — VAL', y_val, nb.predict(X_val))

svm = LinearSVC(C=1.0, class_weight='balanced', random_state=42)
svm.fit(X_train, y_train)
results_val['LinearSVC (balanced)'] = M.print_metrics('LinearSVC (balanced) — VAL', y_val, svm.predict(X_val))

### Validation leaderboard so far

In [ ]:
val_table = M.results_table(results_val)
display(val_table.round(4))
print('\nHeadline = macro_f1. Best model on val:', val_table.index[0])

## 6. Hyperparameter tuning (selected on VALIDATION)
We already have a dedicated val set, so we tune against it directly instead of running
5-fold `GridSearchCV` on 341k rows (much cheaper, and honest). Two quick stages:

**6a. TF-IDF shape** — does adding bigrams / more features help?

In [ ]:
def eval_vectorizer(ngram_range, max_features, model_factory):
    vc, Xtr, Xvl = tf.fit_transform(
        train['text_classical'], val['text_classical'],
        max_features=max_features, ngram_range=ngram_range, min_df=5, max_df=0.9)
    mdl = model_factory()
    mdl.fit(Xtr, y_train)
    m = M.compute_metrics(y_val, mdl.predict(Xvl))
    return m['macro_f1'], vc

lr_factory = lambda: LogisticRegression(max_iter=1000, C=1.0, solver='liblinear',
                                        class_weight='balanced', random_state=42)

grid = [((1, 1), 20000), ((1, 1), 50000), ((1, 2), 50000), ((1, 2), 100000)]
print('ngram   max_feat   val_macroF1')
best = (-1, None, None)
for ng, mf in grid:
    score, _ = eval_vectorizer(ng, mf, lr_factory)
    print(f'{str(ng):7s} {mf:>8,}   {score:.4f}')
    if score > best[0]:
        best = (score, ng, mf)
print(f'\nBest TF-IDF config on val: ngram={best[1]}, max_features={best[2]:,} (macro-F1={best[0]:.4f})')
BEST_NGRAM, BEST_MAXF = best[1], best[2]

**6b. Regularization `C`** for Logistic Regression (with the best TF-IDF config):

In [ ]:
# Rebuild features once with the winning TF-IDF config.
vec, X_train, X_val, X_test = tf.fit_transform(
    train['text_classical'], val['text_classical'], test['text_classical'],
    max_features=BEST_MAXF, ngram_range=BEST_NGRAM, min_df=5, max_df=0.9)

print('C       val_macroF1')
bestC = (-1, None)
for C in [0.1, 0.5, 1.0, 3.0, 10.0]:
    m = LogisticRegression(max_iter=1000, C=C, solver='liblinear',
                           class_weight='balanced', random_state=42)
    m.fit(X_train, y_train)
    f1 = M.compute_metrics(y_val, m.predict(X_val))['macro_f1']
    print(f'{C:<6} {f1:.4f}')
    if f1 > bestC[0]:
        bestC = (f1, C)
BEST_C = bestC[1]
print(f'\nBest C on val: {BEST_C} (macro-F1={bestC[0]:.4f})')

## 7. Feature-engineering bonus — add document-length features
EDA showed humans write **longer & more variable** than AI. We hstack log(char count) and
log(word count) onto TF-IDF and check whether it moves val macro-F1.

In [ ]:
from scipy.sparse import hstack

X_train_len = tf.stack_length(X_train, train['text'])
X_val_len   = tf.stack_length(X_val,   val['text'])

lr_len = LogisticRegression(max_iter=1000, C=BEST_C, solver='liblinear',
                            class_weight='balanced', random_state=42)
lr_len.fit(X_train_len, y_train)
m_len = M.print_metrics('LogReg + length features — VAL', y_val, lr_len.predict(X_val_len))

base_f1 = M.compute_metrics(y_val, LogisticRegression(
    max_iter=1000, C=BEST_C, solver='liblinear', class_weight='balanced',
    random_state=42).fit(X_train, y_train).predict(X_val))['macro_f1']
USE_LENGTH = m_len['macro_f1'] > base_f1
print(f'\nwords-only macro-F1={base_f1:.4f}  vs  +length macro-F1={m_len["macro_f1"]:.4f}')
print('Keep length features?' , USE_LENGTH)

## 8. FINAL — train the chosen model, evaluate ONCE on the test set
This is the sealed test set — the number below is **the baseline the Transformer must beat**.
Standard practice: after selecting on val, refit on **train + val** (all non-test data) before
the final test evaluation.

In [ ]:
from scipy.sparse import vstack

# Refit TF-IDF on train+val combined, using the winning config.
trainval_text = pd.concat([train['text_classical'], val['text_classical']])
trainval_y    = np.concatenate([y_train, y_val])

vec_final = tf.build_vectorizer(max_features=BEST_MAXF, ngram_range=BEST_NGRAM,
                                min_df=5, max_df=0.9)
X_tv   = vec_final.fit_transform(trainval_text)
X_test_final = vec_final.transform(test['text_classical'])

if USE_LENGTH:
    X_tv          = tf.stack_length(X_tv, pd.concat([train['text'], val['text']]))
    X_test_final  = tf.stack_length(X_test_final, test['text'])

final_model = LogisticRegression(max_iter=1000, C=BEST_C, solver='liblinear',
                                 class_weight='balanced', random_state=42)
final_model.fit(X_tv, trainval_y)
y_pred = final_model.predict(X_test_final)

print('#########  BASELINE — TEST SET  #########')
baseline = M.print_metrics('Best classical model — TEST', y_test, y_pred)
print()
print(M.report(y_test, y_pred))

### Confusion matrix on test

In [ ]:
fig = M.plot_confusion(y_test, y_pred, normalize='true',
                       title='Classical Baseline — Test (row-normalized)',
                       savepath=os.path.join(RESULTS, 'confusion_test.png'))
import matplotlib.pyplot as plt
plt.show()
# The bottom-left cell = humans wrongly flagged as AI (the ethically sensitive errors).

## 9. What the model learned — top features per class
Reuse of workshop week 11: the Logistic Regression coefficients tell us which words push
toward **AI** vs **human**. This should echo the EDA (AI = abstract vocabulary; human =
misspellings / concrete nouns) — good for the report's interpretability section.

In [ ]:
names = np.array(vec_final.get_feature_names_out())
# If length features were appended, coefs has 2 extra entries at the end — trim to vocab.
coefs = final_model.coef_[0][:len(names)]

top_ai    = coefs.argsort()[-20:][::-1]     # most positive -> AI (label 1)
top_human = coefs.argsort()[:20]            # most negative -> human (label 0)

print('Top 20 words predicting AI:')
print('  ' + ', '.join(names[top_ai]))
print('\nTop 20 words predicting HUMAN:')
print('  ' + ', '.join(names[top_human]))

## 10. Error analysis (workshop week 11 · Final-Project TODO)
Where does the baseline fail, and do the mistakes share a pattern?

In [ ]:
err = pd.DataFrame({
    'text': test['text'].values,
    'true': y_test,
    'pred': y_pred,
    'words': test['text'].str.split().str.len().values,
})
fp = err[(err.pred == 1) & (err.true == 0)]   # human wrongly called AI
fn = err[(err.pred == 0) & (err.true == 1)]   # AI wrongly called human
print(f'False positives (human->AI): {len(fp):,}')
print(f'False negatives (AI->human): {len(fn):,}')

correct = err[err.pred == err.true]
wrong   = err[err.pred != err.true]
print(f'\navg words — correct: {correct.words.mean():.0f} | misclassified: {wrong.words.mean():.0f}')

print('\n--- 2 humans wrongly flagged as AI (ethically sensitive) ---')
for t in fp['text'].head(2):
    print(' •', t[:200].replace('\n', ' '), '...')
print('\n--- 2 AI texts that slipped through as human ---')
for t in fn['text'].head(2):
    print(' •', t[:200].replace('\n', ' '), '...')

## 11. Save results → `classical_results/` (on Drive)
Download these into the repo's `classical_model/results/` for submission.

In [ ]:
# All models compared on VAL, plus the final TEST row.
final_row = {'FINAL: best classical (TEST)': baseline}
leaderboard = M.results_table({**results_val, **final_row}).round(4)
csv_path = os.path.join(RESULTS, 'metrics_summary.csv')
leaderboard.to_csv(csv_path)
display(leaderboard)
print('[saved]', csv_path)

with open(os.path.join(RESULTS, 'baseline_score.txt'), 'w') as f:
    f.write(f"Classical baseline (TEST)\n")
    f.write(f"model         : LogisticRegression(C={BEST_C}, class_weight=balanced)\n")
    f.write(f"tfidf         : ngram={BEST_NGRAM}, max_features={BEST_MAXF}, length_feats={USE_LENGTH}\n")
    f.write(f"accuracy      : {baseline['accuracy']:.4f}\n")
    f.write(f"macro_f1      : {baseline['macro_f1']:.4f}  <- number the Transformer must beat\n")
    f.write(f"AI  P/R/F1    : {baseline['precision_AI']:.4f}/{baseline['recall_AI']:.4f}/{baseline['f1_AI']:.4f}\n")
print('[saved] baseline_score.txt')

## 12. Summary (fill in after running)

- **Best model:** _e.g. LogReg, C=…, ngram=(1,2), max_features=…, length_feats=…_
- **Baseline test macro-F1:** _____  · accuracy _____ · AI recall _____
- **`class_weight='balanced'` verdict:** _did it raise AI recall / macro-F1?_
- **Length features:** _kept or dropped?_
- **Top signals:** AI ↔ abstract vocab (…); human ↔ misspellings / concrete nouns (…) — matches EDA.
- **Error pattern:** _misclassified texts tend to be shorter/longer? which direction dominates?_
- **Ethics note:** false positives = humans flagged as AI — the direction to watch (careful/non-native writers).

> This macro-F1 is the target for Member B's DistilBERT model in Week 2.